# HALO Harness Optimization

This notebook keeps the existing Databricks agent demo intact, then adds a harness optimization pass with [HALO](https://github.com/context-labs/HALO).

The goal is broader than prompt optimization or skill generation. HALO reviews the same evidence used by skills generation and produces a Codex-ready handoff for improving the whole harness: prompts, tool routing, sufficiency checks, Genie fallback behavior, skills, eval coverage, and deployment guardrails.

Prerequisites:
- Run `00_setup.ipynb` through `09-Evaluation.ipynb` for the existing demo path.
- Run `05-JudgeAlignment.ipynb` so the aligned judge has semantic and episodic memory.
- Run `06-PromptOptimization.ipynb` and `07-AgentSkillsGeneration.ipynb` so optimized prompt and skills artifacts exist.
- Use `gpt-5-4-external` as the model endpoint for HALO. If Databricks OpenAI-compatible routing does not support HALO's Agents SDK calls, this notebook can fall back to direct OpenAI with `OPENAI_API_KEY`.

In [ ]:
import json
from datetime import datetime, timezone

BOOTSTRAP_VOLUME_DIR = "/Volumes/main/at_bat_assistant/agent_skills_gepa/halo"
BOOTSTRAP_STATUS_PATH = f"{BOOTSTRAP_VOLUME_DIR}/run_status.json"
BOOTSTRAP_LOG_PATH = f"{BOOTSTRAP_VOLUME_DIR}/run_log.txt"
payload = {
    "ts": datetime.now(timezone.utc).isoformat(),
    "stage": "bootstrap",
    "message": "Starting HALO dependency install cell",
    "data": {"note": "If this remains the latest status, the notebook is still in %pip install or Python restart."},
}
dbutils.fs.mkdirs(BOOTSTRAP_VOLUME_DIR)
dbutils.fs.put(BOOTSTRAP_STATUS_PATH, json.dumps(payload, indent=2), True)
dbutils.fs.put(BOOTSTRAP_LOG_PATH, json.dumps(payload) + "\n", True)
print(payload)


In [ ]:
%pip install -q --prefer-binary "halo-engine==0.1.10" "mlflow>=3.11.1" "typing_extensions>=4.15.0" "databricks-agents>=1.6.0" openai


In [ ]:
from __future__ import annotations

import json
import os
import re
import time
import traceback
import uuid
import hashlib
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import sys
sys.modules.pop("typing_extensions", None)

import mlflow
from databricks.sdk import WorkspaceClient
from IPython.display import Markdown, display

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())
CATALOG = CONFIG["workspace"]["catalog"]
SCHEMA = CONFIG["workspace"]["schema"]
EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
ALIGNED_JUDGE_NAME = CONFIG["judges"]["aligned_judge_name"]
SKILLS_VOLUME_PATH = CONFIG["skills"]["gepa_volume_path"]
UC_TOOL_NAMES = CONFIG["tools"]["uc_tool_names"]
HALO_MODEL = os.getenv("HALO_MODEL", CONFIG["llm"].get("endpoint_name", "gpt-5-4-external")).replace("databricks:/", "")
HALO_DIRECT_OPENAI_MODEL = os.getenv("HALO_DIRECT_OPENAI_MODEL", "gpt-5.4-nano")

# Prompt registry operations use service-principal OAuth in the earlier notebooks.
# Keep the same auth pattern here so HALO receives the actual optimized prompt text.
_prompt_auth = CONFIG.get("prompt_registry_auth", {})
if _prompt_auth.get("databricks_host") and not os.getenv("DATABRICKS_HOST"):
    os.environ["DATABRICKS_HOST"] = _prompt_auth["databricks_host"].rstrip("/")

try:
    _PROMPT_SP_ID = dbutils.secrets.get(scope=_prompt_auth["secret_scope_name"], key=_prompt_auth["oauth_client_id_key"])
    _PROMPT_SP_SECRET = dbutils.secrets.get(scope=_prompt_auth["secret_scope_name"], key=_prompt_auth["oauth_client_secret_key"])
except Exception as exc:
    _PROMPT_SP_ID = None
    _PROMPT_SP_SECRET = None
    print(f"Prompt registry OAuth credentials unavailable; will try ambient auth and checkpoint fallback. Reason: {type(exc).__name__}: {exc}")

@contextmanager
def _prompt_registry_auth():
    saved_token = os.environ.pop("DATABRICKS_TOKEN", None)
    saved_client_id = os.environ.get("DATABRICKS_CLIENT_ID")
    saved_client_secret = os.environ.get("DATABRICKS_CLIENT_SECRET")
    if _PROMPT_SP_ID and _PROMPT_SP_SECRET:
        os.environ["DATABRICKS_CLIENT_ID"] = _PROMPT_SP_ID
        os.environ["DATABRICKS_CLIENT_SECRET"] = _PROMPT_SP_SECRET
    try:
        yield
    finally:
        if saved_client_id is None:
            os.environ.pop("DATABRICKS_CLIENT_ID", None)
        else:
            os.environ["DATABRICKS_CLIENT_ID"] = saved_client_id
        if saved_client_secret is None:
            os.environ.pop("DATABRICKS_CLIENT_SECRET", None)
        else:
            os.environ["DATABRICKS_CLIENT_SECRET"] = saved_client_secret
        if saved_token is not None:
            os.environ["DATABRICKS_TOKEN"] = saved_token

os.environ.setdefault("MLFLOW_TRACKING_URI", "databricks")
os.environ.setdefault("MLFLOW_REGISTRY_URI", f"databricks-uc://{CATALOG}")
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri(f"databricks-uc://{CATALOG}")
try:
    mlflow.set_experiment(experiment_id=EXPERIMENT_ID)
except Exception as exc:
    print(f"Skipping active experiment setup; trace search will use explicit experiment location. Reason: {exc}")
w = WorkspaceClient()
ARTIFACT_ROOT = Path("/tmp/atbat_halo")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
HALO_TRACE_PATH = ARTIFACT_ROOT / "halo_traces.jsonl"
HALO_OUTPUT_JSON = ARTIFACT_ROOT / "halo_output.json"
HALO_HANDOFF_MD = ARTIFACT_ROOT / "codex_handoff.md"
HALO_LOG_PATH = ARTIFACT_ROOT / "run_log.txt"
HALO_STATUS_JSON = ARTIFACT_ROOT / "run_status.json"
HALO_VOLUME_DIR = f"{SKILLS_VOLUME_PATH.rstrip('/')}/halo"
HALO_VOLUME_LOG_PATH = f"{HALO_VOLUME_DIR}/run_log.txt"
HALO_VOLUME_STATUS_PATH = f"{HALO_VOLUME_DIR}/run_status.json"
HALO_LOG_PATH.write_text("", encoding="utf-8")

def log_event(stage: str, message: str, **data: Any) -> None:
    payload = {
        "ts": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "message": message,
        "data": data,
    }
    line = json.dumps(payload, default=str, ensure_ascii=False)
    print(f"[{payload['ts']}] {stage}: {message} {data if data else ''}")
    with HALO_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
    HALO_STATUS_JSON.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    try:
        dbutils.fs.mkdirs(HALO_VOLUME_DIR)
        dbutils.fs.put(HALO_VOLUME_LOG_PATH, HALO_LOG_PATH.read_text(encoding="utf-8"), True)
        dbutils.fs.put(HALO_VOLUME_STATUS_PATH, HALO_STATUS_JSON.read_text(encoding="utf-8"), True)
    except Exception as exc:
        print(f"Could not persist live HALO log: {type(exc).__name__}: {exc}")

print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"HALO Databricks model endpoint: {HALO_MODEL}")
print(f"HALO direct OpenAI fallback model: {HALO_DIRECT_OPENAI_MODEL}")
print(f"Artifact root: {ARTIFACT_ROOT}")
log_event("init", "HALO notebook initialized", experiment_id=EXPERIMENT_ID, halo_model=HALO_MODEL, artifact_root=str(ARTIFACT_ROOT), volume_dir=HALO_VOLUME_DIR)

## Load The Same Evidence Used For Skills Generation

HALO should optimize the harness from the same evidence set used by `07-AgentSkillsGeneration.ipynb`: UC function signatures, optimized prompt, aligned judge memory, evaluated traces, and generated skills.

In [ ]:
def _safe_json(value: Any, max_chars: int = 20000) -> str:
    try:
        text = json.dumps(value, default=str, ensure_ascii=False)
    except Exception:
        text = str(value)
    if len(text) > max_chars:
        return text[:max_chars] + f"... [truncated {len(text) - max_chars} chars]"
    return text


def _read_volume_text(path: str) -> str:
    try:
        return Path(path).read_text()
    except Exception:
        try:
            return dbutils.fs.head(path, 200000)
        except Exception:
            return ""


def _list_volume_files(path: str, *, recursive: bool = False, exclude_prefixes: tuple[str, ...] = ()) -> list[str]:
    def _is_excluded(candidate: str) -> bool:
        return any(candidate.rstrip('/').startswith(prefix.rstrip('/')) for prefix in exclude_prefixes)

    try:
        entries = dbutils.fs.ls(path)
    except Exception:
        return []

    files = []
    for entry in entries:
        entry_path = entry.path
        if _is_excluded(entry_path):
            continue
        if entry_path.endswith('/'):
            if recursive:
                files.extend(_list_volume_files(entry_path, recursive=True, exclude_prefixes=exclude_prefixes))
        else:
            files.append(entry_path)
    return files


def _extract_user_query(trace) -> str:
    try:
        request = trace.data.request
        if isinstance(request, str):
            request = json.loads(request)
        inputs = request.get("input", request.get("inputs", [])) if isinstance(request, dict) else []
        if isinstance(inputs, dict):
            inputs = inputs.get("input", [])
        if isinstance(inputs, list):
            for msg in inputs:
                if isinstance(msg, dict) and msg.get("role") == "user":
                    return str(msg.get("content", ""))
    except Exception:
        pass
    return ""


def _extract_agent_response(trace) -> str:
    try:
        response = trace.data.response
        if isinstance(response, str):
            response = json.loads(response)
        output = response.get("output", []) if isinstance(response, dict) else []
        texts = []
        for item in output if isinstance(output, list) else []:
            if not isinstance(item, dict):
                continue
            for content in item.get("content", []) or []:
                if isinstance(content, dict) and content.get("type") == "output_text":
                    texts.append(str(content.get("text", "")))
        return "\n".join(t for t in texts if t)
    except Exception:
        return ""


def _extract_tool_calls(trace) -> list[dict[str, Any]]:
    try:
        response = trace.data.response
        if isinstance(response, str):
            response = json.loads(response)
        output = response.get("output", []) if isinstance(response, dict) else []
    except Exception:
        output = []
    calls_by_id = {}
    ordered = []
    for item in output if isinstance(output, list) else []:
        if not isinstance(item, dict):
            continue
        if item.get("type") == "function_call":
            call_id = item.get("call_id") or item.get("id") or str(len(ordered))
            entry = {
                "call_id": call_id,
                "name": item.get("name"),
                "arguments": item.get("arguments"),
                "output": None,
            }
            calls_by_id[call_id] = entry
            ordered.append(entry)
        elif item.get("type") == "function_call_output":
            call_id = item.get("call_id")
            if call_id in calls_by_id:
                calls_by_id[call_id]["output"] = item.get("output")
    return ordered


def _assessment_summary(trace) -> list[dict[str, Any]]:
    rows = []
    for assessment in getattr(trace.info, "assessments", []) or []:
        feedback = getattr(assessment, "feedback", None)
        source = getattr(getattr(assessment, "source", None), "source_type", None)
        rows.append({
            "name": getattr(assessment, "name", None),
            "source": source,
            "signal_source": _assessment_signal_source(source),
            "value": getattr(feedback, "value", None) if feedback else None,
            "rationale": getattr(assessment, "rationale", None),
        })
    return rows


def _assessment_signal_source(source: Any) -> str:
    text = str(source or "").lower()
    if "human" in text:
        return "human_feedback"
    if "llm" in text or "judge" in text:
        return "llm_judge_feedback"
    if "code" in text or "metric" in text or "heuristic" in text:
        return "deterministic_eval_feedback"
    return "mlflow_assessment_feedback"


def _compact_preview(value: Any, max_chars: int = 1200) -> str:
    text = value if isinstance(value, str) else _safe_json(value, max_chars=max_chars * 2)
    text = text.strip()
    return text[:max_chars] + ("..." if len(text) > max_chars else "")



def _extract_prompt_text(prompt_obj: Any) -> str:
    template = getattr(prompt_obj, "template", None)
    if template:
        return str(template)
    if hasattr(prompt_obj, "format"):
        try:
            return str(prompt_obj.format())
        except Exception:
            pass
    return str(prompt_obj)


def _load_latest_gepa_prompt_from_checkpoint(prompt_name: str) -> tuple[str | None, dict[str, Any]]:
    """Fallback for HALO: read the optimized prompt text from GEPA checkpoint tables."""
    table_prefix = f"{CATALOG}.{SCHEMA}.gepa_experiment_checkpoint_"
    try:
        tables = [row.tableName for row in spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()]
    except Exception as exc:
        return None, {"source": "gepa_checkpoint", "loaded": False, "error": f"SHOW TABLES failed: {exc}"}

    candidates = [name for name in tables if name.startswith("gepa_experiment_checkpoint_")]
    best = None
    best_meta = {"source": "gepa_checkpoint", "loaded": False, "candidate_tables": candidates}
    for table_name in candidates:
        fqtn = f"{CATALOG}.{SCHEMA}.{table_name}"
        try:
            rows = spark.table(fqtn).collect()
        except Exception as exc:
            continue
        for row in rows:
            data = row.asDict(recursive=True)
            final_score = data.get("final_score") or 0
            prompt_text = None
            templates_json = data.get("prompt_templates_json")
            if templates_json:
                try:
                    templates = json.loads(templates_json)
                    prompt_text = templates.get(prompt_name)
                except Exception:
                    prompt_text = None
            if not prompt_text:
                prompt_text = data.get("prompt_template")
            if not prompt_text:
                continue
            candidate = (float(final_score), len(prompt_text), prompt_text, fqtn, data)
            if best is None or candidate[:2] > best[:2]:
                best = candidate
    if best is None:
        return None, best_meta
    final_score, _, prompt_text, fqtn, data = best
    return prompt_text, {
        "source": "gepa_checkpoint",
        "loaded": True,
        "table": fqtn,
        "run_idx": data.get("run_idx"),
        "agent_type": data.get("agent_type"),
        "initial_score": data.get("initial_score"),
        "final_score": data.get("final_score"),
    }


def _load_optimized_prompt_text(prompt_name: str) -> tuple[str, dict[str, Any]]:
    attempts = []
    try:
        with _prompt_registry_auth():
            prompt_obj = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@production")
        text = _extract_prompt_text(prompt_obj)
        return text, {
            "source": "prompt_registry_production",
            "loaded": True,
            "prompt_version": str(getattr(prompt_obj, "version", "")),
            "prompt_hash": hashlib.md5(text.encode()).hexdigest()[:12],
        }
    except Exception as exc:
        attempts.append({"source": "prompt_registry_production", "loaded": False, "error": str(exc)[:1000]})

    text, meta = _load_latest_gepa_prompt_from_checkpoint(prompt_name)
    attempts.append(meta)
    if text:
        meta = {**meta, "prompt_hash": hashlib.md5(text.encode()).hexdigest()[:12], "attempts": attempts}
        return text, meta

    raise RuntimeError(f"Could not load optimized prompt from prompt registry or GEPA checkpoint tables: {attempts}")


def _load_latest_skill_files(skills_volume_path: str) -> tuple[list[dict[str, str]], dict[str, Any]]:
    """Load only the current skill set described by the latest manifest, not examples/gotchas/history."""
    manifest_path = f"{skills_volume_path.rstrip('/')}/_manifest.json"
    manifest = {}
    try:
        manifest = json.loads(_read_volume_text(manifest_path))
    except Exception as exc:
        manifest = {"load_error": str(exc)}

    loaded = []
    missing = []
    for item in manifest.get("skills", []) or []:
        name = item.get("name") or Path(item.get("filename", "")).stem
        filename = item.get("filename") or f"{name}.md"
        candidate_paths = [
            f"{skills_volume_path.rstrip('/')}/{name}/skill.md",
            f"{skills_volume_path.rstrip('/')}/{filename}",
        ]
        content = ""
        selected = None
        for candidate in candidate_paths:
            content = _read_volume_text(candidate)
            if content:
                selected = candidate
                break
        if selected:
            loaded.append({"path": selected, "name": name, "content": content[:30000]})
        else:
            missing.append({"name": name, "candidate_paths": candidate_paths})

    meta = {
        "manifest_path": manifest_path,
        "generated_at": manifest.get("generated_at"),
        "manifest_skill_count": len(manifest.get("skills", []) or []),
        "loaded_skill_count": len(loaded),
        "missing": missing,
    }
    return loaded, meta


In [ ]:
log_event("evidence", "Loading UC function descriptions", function_count=len(UC_TOOL_NAMES))
# UC function signatures
uc_functions = []
for fn in UC_TOOL_NAMES:
    try:
        rows = spark.sql(f"DESCRIBE FUNCTION EXTENDED {fn}").collect()
        uc_functions.append({"name": fn, "description": "\n".join(str(r[0]) for r in rows)})
    except Exception as exc:
        uc_functions.append({"name": fn, "description": f"ERROR loading signature: {exc}"})
uc_functions_text = "\n\n".join(f"### {f['name']}\n{f['description']}" for f in uc_functions)
log_event("evidence", "Loaded UC function descriptions", loaded=len(uc_functions), errors=sum(1 for f in uc_functions if f['description'].startswith('ERROR')))

log_event("evidence", "Loading optimized production prompt", prompt_name=PROMPT_NAME)
optimized_prompt_text, optimized_prompt_meta = _load_optimized_prompt_text(PROMPT_NAME)
log_event(
    "evidence",
    "Loaded optimized prompt",
    chars=len(optimized_prompt_text),
    source=optimized_prompt_meta.get("source"),
    prompt_version=optimized_prompt_meta.get("prompt_version"),
    prompt_hash=optimized_prompt_meta.get("prompt_hash"),
    load_failed=False,
)

log_event("evidence", "Loading aligned judge memory", judge_name=ALIGNED_JUDGE_NAME)
# Aligned judge memory. HALO can still run if the local MLflow wheel does not expose mlflow.genai.scorers.
semantic_memory = []
episodic_count = None
try:
    from mlflow.genai.scorers import get_scorer

    aligned_judge = get_scorer(name=ALIGNED_JUDGE_NAME, experiment_id=EXPERIMENT_ID)
    try:
        _ = aligned_judge(
            inputs={"input": [{"role": "user", "content": "How should a hitter approach a pitcher with runners on base?"}]},
            outputs={"response": "Use count, handedness, pitch mix, and location evidence before making a recommendation."},
        )
    except Exception as exc:
        print(f"Judge warmup raised {type(exc).__name__}; continuing with loaded metadata")

    for g in getattr(aligned_judge, "_semantic_memory", []) or []:
        semantic_memory.append({"guideline": g.guideline_text, "source_trace_ids": g.source_trace_ids})
    episodic_count = len(getattr(aligned_judge, "_episodic_memory", []) or [])
except Exception as exc:
    log_event("evidence", "Skipped direct aligned judge memory load", error_type=type(exc).__name__, error=str(exc)[:1000])
    print("HALO will use trace assessments and evaluation metrics as judge evidence instead.")
log_event("evidence", "Aligned judge memory load complete", semantic_guidelines=len(semantic_memory), episodic_examples=episodic_count)

log_event("evidence", "Loading latest generated skills from UC volume manifest", skills_volume=SKILLS_VOLUME_PATH)
skill_files, skill_load_meta = _load_latest_skill_files(SKILLS_VOLUME_PATH)
log_event(
    "evidence",
    "Loaded latest generated skill artifacts",
    skill_file_count=len(skill_files),
    manifest_generated_at=skill_load_meta.get("generated_at"),
    manifest_skill_count=skill_load_meta.get("manifest_skill_count"),
    missing_skill_count=len(skill_load_meta.get("missing", [])),
    skill_paths=[s["path"] for s in skill_files],
)
if skill_load_meta.get("missing"):
    print("Missing skills from manifest:", json.dumps(skill_load_meta["missing"], indent=2))

## Build HALO-Compatible Trace JSONL

HALO expects canonical span JSONL. This cell converts MLflow evaluation traces into one root span plus one span per tool call, preserving user query, response, assessments, tool arguments, and tool outputs.

In [ ]:
def _iso_now() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _stable_hex(value: str, length: int) -> str:
    import hashlib

    return hashlib.sha256(value.encode("utf-8")).hexdigest()[:length]


def _synthetic_trace_id(value: str) -> str:
    return _stable_hex(f"halo-context:{value}", 32)


def _synthetic_span_id(value: str) -> str:
    return _stable_hex(f"halo-span:{value}", 16)


def _span(trace_id: str, name: str, attrs: dict[str, Any], parent_span_id: str = "") -> dict[str, Any]:
    sid = uuid.uuid4().hex[:16]
    return {
        "trace_id": trace_id.replace("tr-", "")[:32].ljust(32, "0"),
        "span_id": sid,
        "parent_span_id": parent_span_id,
        "trace_state": "",
        "name": name,
        "kind": "SPAN_KIND_INTERNAL",
        "start_time": _iso_now(),
        "end_time": _iso_now(),
        "status": {"code": "STATUS_CODE_OK", "message": ""},
        "resource": {"attributes": {"service.name": "at-bat-assistant", "project.id": "at-bat-assistant-cais"}},
        "scope": {"name": "mlflow-to-halo", "version": "2"},
        "attributes": attrs,
    }


def _context_span(name: str, signal_source: str, attrs: dict[str, Any], trace_key: str | None = None) -> dict[str, Any]:
    trace_id = _synthetic_trace_id(trace_key or name)
    span = _span(trace_id, name, {
        "openinference.span.kind": signal_source.upper(),
        "inference.export.schema_version": 1,
        "inference.project_id": "at_bat_assistant_halo_optimization_context",
        "inference.observation_kind": signal_source,
        "optimizer.signal_source": signal_source,
        **attrs,
    })
    span["span_id"] = _synthetic_span_id(f"{name}:{signal_source}:{trace_key or ''}")
    return span


def _halo_input_summary_rows() -> list[tuple[str, int, str, str]]:
    assessment_counts = {}
    for trace in trace_summaries:
        for assessment in trace.get("assessments", []):
            source = assessment.get("signal_source", "mlflow_assessment_feedback")
            assessment_counts[source] = assessment_counts.get(source, 0) + 1
    return [
        ("Runtime traces", len(trace_summaries), "original MLflow traces", "user query, agent response, tool calls, tool outputs, trace-level assessments"),
        ("Tool call spans", sum(t.get("tool_count", 0) for t in trace_summaries), "child spans on runtime traces", "UC function names, arguments, outputs, timeout and BAD_REQUEST evidence"),
        ("Human feedback", assessment_counts.get("human_feedback", 0), "assessment spans attached to source traces", "human review rationale when present"),
        ("LLM judge feedback", assessment_counts.get("llm_judge_feedback", 0), "assessment spans attached to source traces", "judge scores, rationales, and pass/fail signals"),
        ("Deterministic eval feedback", assessment_counts.get("deterministic_eval_feedback", 0), "assessment spans attached to source traces", "code or metric-backed assessment signals"),
        ("Other MLflow assessment feedback", assessment_counts.get("mlflow_assessment_feedback", 0), "assessment spans attached to source traces", "assessment values that do not expose a more specific source type"),
        ("Aligned judge memory", len(semantic_memory), "global context spans", "semantic guidelines and episodic memory count from judge alignment"),
        ("Generated skills", len(skill_files), "global context spans", "skill markdown/json artifacts produced by the skills generation loop"),
        ("Optimized prompt", 1, "global context span", "optimized prompt text loaded from MLflow prompt registry or GEPA checkpoint fallback"),
        ("UC function signatures", len(uc_functions), "global context spans", "DESCRIBE FUNCTION EXTENDED output for assistant tools"),
    ]


def render_halo_input_summary() -> str:
    rows = _halo_input_summary_rows()
    lines = [
        "### HALO input summary",
        "",
        "| Input signal | Count | Where it lives | What is included |",
        "| --- | ---: | --- | --- |",
    ]
    lines.extend(f"| {name} | {count} | {location} | {included} |" for name, count, location, included in rows)
    return "\n".join(lines)


log_event("traces", "Searching for aligned traces", experiment_id=EXPERIMENT_ID, filter="tag.align = 'use'")
# Prefer the aligned traces if present because these are the examples that drive the harness loop.
traces = mlflow.search_traces(
    locations=[EXPERIMENT_ID],
    filter_string="tag.align = 'use'",
    return_type="list",
)
log_event("traces", "Aligned trace search complete", trace_count=len(traces))
if not traces:
    log_event("traces", "No aligned traces found; falling back to eval-complete traces", filter="tag.eval = 'complete'")
    traces = mlflow.search_traces(
        locations=[EXPERIMENT_ID],
        filter_string="tag.eval = 'complete'",
        return_type="list",
    )
log_event("traces", "Loaded traces for HALO", trace_count=len(traces))

log_event("traces", "Converting MLflow traces and source-labeled context to HALO span JSONL")
span_rows = []
trace_summaries = []
for trace in traces:
    tid = trace.info.trace_id
    query = _extract_user_query(trace)
    response = _extract_agent_response(trace)
    tools = _extract_tool_calls(trace)
    assessments = _assessment_summary(trace)
    root = _span(tid, "agent_evaluation_trace", {
        "openinference.span.kind": "AGENT",
        "inference.export.schema_version": 1,
        "inference.project_id": "at_bat_assistant_halo_optimization_context",
        "inference.observation_kind": "runtime_trace",
        "optimizer.signal_source": "runtime_trace",
        "input.value": query,
        "output.value": response,
        "atbat.trace_id": tid,
        "atbat.user_query": query,
        "atbat.agent_response": response,
        "atbat.assessments": _safe_json(assessments),
        "atbat.tool_call_count": len(tools),
        "atbat.tool_calls": _safe_json(tools),
    })
    span_rows.append(root)
    trace_summaries.append({
        "trace_id": tid,
        "query": query,
        "response_preview": response[:500],
        "tool_count": len(tools),
        "assessments": assessments,
    })
    for i, tool in enumerate(tools, 1):
        status = "STATUS_CODE_ERROR" if str(tool.get("output", "")).lower().startswith("error") else "STATUS_CODE_OK"
        child = _span(tid, f"tool_call_{i}:{tool.get('name')}", {
            "openinference.span.kind": "TOOL",
            "inference.export.schema_version": 1,
            "inference.project_id": "at_bat_assistant_halo_optimization_context",
            "inference.observation_kind": "runtime_tool_call",
            "optimizer.signal_source": "runtime_trace",
            "tool.name": tool.get("name"),
            "tool.arguments": tool.get("arguments"),
            "tool.output": str(tool.get("output", ""))[:30000],
            "atbat.trace_id": tid,
        }, parent_span_id=root["span_id"])
        child["status"]["code"] = status
        span_rows.append(child)
    for i, assessment in enumerate(assessments, 1):
        signal_source = assessment.get("signal_source", "mlflow_assessment_feedback")
        span_rows.append(_span(tid, f"assessment_{i}:{assessment.get('name')}", {
            "openinference.span.kind": signal_source.upper(),
            "inference.export.schema_version": 1,
            "inference.project_id": "at_bat_assistant_halo_optimization_context",
            "inference.observation_kind": signal_source,
            "optimizer.signal_source": signal_source,
            "assessment.name": assessment.get("name"),
            "assessment.source": assessment.get("source"),
            "assessment.value": _safe_json(assessment.get("value")),
            "assessment.rationale": assessment.get("rationale"),
            "atbat.trace_id": tid,
        }, parent_span_id=root["span_id"]))

span_rows.append(_context_span("harness.config", "harness_config", {
    "harness.catalog": CATALOG,
    "harness.schema": SCHEMA,
    "harness.prompt_name": PROMPT_NAME,
    "harness.aligned_judge_name": ALIGNED_JUDGE_NAME,
    "harness.model_endpoint": HALO_MODEL,
    "harness.notebook_range": "00_setup through 10-HALOHarnessOptimization",
}))
span_rows.append(_context_span("optimized_prompt.production", "optimized_prompt", {
    "prompt.name": PROMPT_NAME,
    "prompt.text": optimized_prompt_text[:50000],
}))
span_rows.append(_context_span("aligned_judge.memory", "aligned_judge_memory", {
    "judge.name": ALIGNED_JUDGE_NAME,
    "judge.semantic_guidelines": _safe_json(semantic_memory, max_chars=50000),
    "judge.episodic_memory_count": episodic_count,
}))
for index, fn in enumerate(uc_functions):
    span_rows.append(_context_span(f"uc_function.signature:{fn.get('name')}", "uc_function_signatures", {
        "uc_function.name": fn.get("name"),
        "uc_function.description": fn.get("description"),
    }, trace_key=f"uc-function-{index}-{fn.get('name')}"))
for index, skill in enumerate(skill_files):
    span_rows.append(_context_span(f"generated_skill.artifact:{index}", "generated_skills", {
        "skill.path": skill.get("path"),
        "skill.content": skill.get("content"),
    }, trace_key=f"generated-skill-{index}-{skill.get('path')}"))
span_rows.append(_context_span("trace_set.summary", "trace_set_summary", {
    "trace_set.trace_count": len(trace_summaries),
    "trace_set.span_count_before_summary": len(span_rows),
    "trace_set.summary": _safe_json(trace_summaries, max_chars=80000),
}))

with HALO_TRACE_PATH.open("w") as f:
    for row in span_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

input_summary = render_halo_input_summary()
display(Markdown(input_summary))
log_event("traces", "Wrote source-labeled HALO span JSONL", span_count=len(span_rows), trace_count=len(trace_summaries), trace_path=str(HALO_TRACE_PATH))
try:
    dbutils.fs.put(f"{HALO_VOLUME_DIR}/halo_traces.in_progress.jsonl", HALO_TRACE_PATH.read_text(encoding="utf-8"), True)
    dbutils.fs.put(f"{HALO_VOLUME_DIR}/halo_input_summary.md", input_summary, True)
    log_event("traces", "Copied in-progress trace JSONL and input summary to UC volume", volume_path=f"{HALO_VOLUME_DIR}/halo_traces.in_progress.jsonl")
except Exception as exc:
    log_event("traces", "Could not copy in-progress trace JSONL", error_type=type(exc).__name__, error=str(exc)[:1000])


### HALO input summary

| Input signal | Count | Where it lives | What is included |
| --- | ---: | --- | --- |
| Runtime traces | 19 | original MLflow traces | user query, agent response, tool calls, tool outputs, trace-level assessments |
| Tool call spans | 105 | child spans on runtime traces | UC function names, arguments, outputs, timeout and BAD_REQUEST evidence |
| Human feedback | 19 | assessment spans attached to source traces | human review rationale when present |
| LLM judge feedback | 57 | assessment spans attached to source traces | judge scores, rationales, and pass/fail signals |
| Deterministic eval feedback | 0 | assessment spans attached to source traces | code or metric-backed assessment signals |
| Other MLflow assessment feedback | 0 | assessment spans attached to source traces | assessment values that do not expose a more specific source type |
| Aligned judge memory | 0 | global context spans | semantic guidelines and episodic memory count from judge alignment |
| Generated skills | 7 | global context spans | skill markdown/json artifacts produced by the skills generation loop |
| Optimized prompt | 1 | global context span | optimized prompt text loaded from MLflow prompt registry or GEPA checkpoint fallback |
| UC function signatures | 7 | global context spans | DESCRIBE FUNCTION EXTENDED output for assistant tools |

## Run HALO

The Databricks endpoint is tried first through OpenAI-compatible routing. If that fails and `OPENAI_API_KEY` is available, the notebook retries against direct OpenAI.

In [ ]:
log_event("halo", "Importing HALO engine modules")
from engine.agents.agent_config import AgentConfig
from engine.engine_config import EngineConfig
from engine.main import run_engine_async
from engine.model_config import ModelConfig
from engine.model_provider_config import ModelProviderConfig
from engine.models.messages import AgentMessage


def _workspace_token() -> str:
    try:
        return dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    except Exception:
        return w.config.authenticate()


def _make_engine_config(model_name: str, *, provider: ModelProviderConfig | None = None) -> EngineConfig:
    root_turns = int(os.getenv("HALO_ROOT_MAX_TURNS", "10"))
    sub_turns = int(os.getenv("HALO_SUB_MAX_TURNS", "6"))
    max_depth = int(os.getenv("HALO_MAX_DEPTH", "1"))
    max_parallel = int(os.getenv("HALO_MAX_PARALLEL_SUBAGENTS", "2"))
    reasoning_effort = os.getenv("HALO_REASONING_EFFORT", "low")
    root_model = ModelConfig(name=model_name, reasoning_effort=reasoning_effort, maximum_output_tokens=16000)
    sub_model = ModelConfig(name=model_name, reasoning_effort=reasoning_effort, maximum_output_tokens=12000)
    synthesis_model = ModelConfig(name=model_name, reasoning_effort=reasoning_effort, maximum_output_tokens=16000)
    compact_model = ModelConfig(name=model_name, maximum_output_tokens=6000)
    return EngineConfig(
        root_agent=AgentConfig(name="halo-root", model=root_model, maximum_turns=root_turns),
        subagent=AgentConfig(name="halo-sub", model=sub_model, maximum_turns=sub_turns),
        synthesis_model=synthesis_model,
        compaction_model=compact_model,
        model_provider=provider or ModelProviderConfig(),
        maximum_depth=max_depth,
        maximum_parallel_subagents=max_parallel,
    )


log_event("halo", "Building source-labeled HALO context", trace_count=len(trace_summaries), skill_count=len(skill_files), semantic_guidelines=len(semantic_memory))
halo_context = {
    "input_summary": _halo_input_summary_rows(),
    "uc_functions": [{"name": f.get("name"), "description_preview": _compact_preview(f.get("description"))} for f in uc_functions],
    "optimized_prompt": _compact_preview(optimized_prompt_text, max_chars=5000),
    "aligned_judge_memory": {
        "semantic_memory": semantic_memory,
        "episodic_memory_count": episodic_count,
    },
    "generated_skills": [{"path": s.get("path"), "content_preview": _compact_preview(s.get("content"))} for s in skill_files],
    "trace_summaries": trace_summaries,
    "source_labels": {
        "runtime_trace": "MLflow execution traces: user inputs, agent outputs, tool calls, and tool results.",
        "human_feedback": "Human-authored MLflow assessment feedback where available.",
        "llm_judge_feedback": "LLM judge assessment scores and rationales from evaluation and judge alignment.",
        "deterministic_eval_feedback": "Code or metric-backed assessment feedback where available.",
        "mlflow_assessment_feedback": "MLflow assessments whose source type is not more specific.",
        "optimized_prompt": "Optimized prompt loaded from MLflow prompt registry or GEPA checkpoint fallback.",
        "uc_function_signatures": "Unity Catalog function descriptions and schemas.",
        "aligned_judge_memory": "Semantic and episodic memory from the aligned judge.",
        "generated_skills": "Skills generated by the prior skills-generation loop.",
        "harness_config": "Notebook/config context for the Databricks at-bat assistant harness.",
    },
}

halo_prompt = f"""
Analyze the Databricks at-bat assistant optimization context as the central source of truth.
The JSONL trace file contains source-labeled spans for runtime traces, tool calls, MLflow assessments, aligned judge memory, generated skills, UC function signatures, optimized prompt text, and harness config.
Treat every recommendation as a harness change, not a one-off answer correction.

The source labels are:
{_safe_json(halo_context['source_labels'], max_chars=12000)}

Input summary:
{render_halo_input_summary()}

Additional compact context:
{_safe_json(halo_context, max_chars=90000)}

Before recommending a change, compare the evidence against the current harness context and distinguish:
- a requirement that is missing from the harness,
- a requirement already present but not reliably followed in execution, and
- an implementation or observability defect.

Write an implementation-first Codex handoff in this exact top-level order:
1. `## Executive summary`
2. `## Top 3 changes to implement first`
3. `## Ranked recommendation table`
4. `## Supporting diagnosis and evidence`
5. `## Detailed recommendations`
6. `## Insights by feedback source`
7. `## Machine-readable summary`

Section requirements:
- `## Executive summary`: briefly state what the current harness already does well, what the highest-value remaining gaps are, and whether the available eval/judge signals indicate the current gate passed.
- `## Top 3 changes to implement first`: list the three most valuable implementation moves with concise rationale.
- `## Ranked recommendation table`: include rank, recommendation, impact, confidence, implementation effort, evidence, and validation.
- `## Supporting diagnosis and evidence`: include recurring harness-level failure modes, classify each against the current harness as missing requirement vs already-present-but-not-reliably-followed vs implementation/observability defect, and state the evidence source for each.
- `## Detailed recommendations`: use these exact subsection headings in this order:
  - `### Behavior contract`
    - `#### Prompt`
    - `#### Skills`
  - `### Runtime implementation`
    - `#### Tools`
    - `#### Control flow`
    - `#### Routing`
  - `### Output contract`
    - `#### Response schema`
  - `### Observability and evals`
    - `#### Observability`
    - `#### Evals`
- `## Insights by feedback source`: summarize what came from each available source label. Include rows or bullets for runtime traces, human feedback, LLM judge feedback, deterministic/MLflow assessment feedback, optimized prompt, UC function signatures, aligned judge memory, generated skills, and harness config. If a source has zero examples, say so explicitly instead of omitting it.
- `## Machine-readable summary`: include one fenced JSON block with `top_priorities`, `evidence_sources`, `validation_plan`, and `manual_review_required`.

Do not add extra top-level sections outside that order. Do not write executable code. Make the handoff concrete enough that Codex can propose notebook and harness patches from it.
Reference these notebook targets where relevant:
- notebooks/03_create_agent_definition.ipynb
- notebooks/06-PromptOptimization.ipynb
- notebooks/07-AgentSkillsGeneration.ipynb
- notebooks/08_create_agent_with_skills.ipynb
- notebooks/09-Evaluation.ipynb
- notebooks/10-HALOHarnessOptimization.ipynb
"""

log_event("halo", "HALO prompt prepared", prompt_chars=len(halo_prompt), trace_path=str(HALO_TRACE_PATH))

async def _run_halo_with_databricks():
    log_event("halo", "Preparing Databricks model provider", endpoint=HALO_MODEL, base_url=f"{w.config.host.rstrip('/')}/serving-endpoints/")
    provider = ModelProviderConfig(
        base_url=f"{w.config.host.rstrip('/')}/serving-endpoints/",
        api_key=_workspace_token(),
    )
    cfg = _make_engine_config(HALO_MODEL, provider=provider)
    log_event("halo", "Starting HALO run via Databricks", endpoint=HALO_MODEL, reasoning_effort=os.getenv("HALO_REASONING_EFFORT", "low"), root_turns=os.getenv("HALO_ROOT_MAX_TURNS", "10"), sub_turns=os.getenv("HALO_SUB_MAX_TURNS", "6"), max_depth=os.getenv("HALO_MAX_DEPTH", "1"), max_parallel=os.getenv("HALO_MAX_PARALLEL_SUBAGENTS", "2"))
    return await run_engine_async([AgentMessage(role="user", content=halo_prompt)], cfg, HALO_TRACE_PATH, telemetry=False)


async def _run_halo_with_direct_openai():
    log_event("halo", "Preparing direct OpenAI fallback", model=HALO_DIRECT_OPENAI_MODEL)
    if not os.getenv("OPENAI_API_KEY"):
        try:
            openai_secret_scope = os.getenv("OPENAI_SECRET_SCOPE", "<your-secret-scope>")
            os.environ["OPENAI_API_KEY"] = dbutils.secrets.get(scope=openai_secret_scope, key="openai_api_key")
        except Exception:
            pass
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Databricks HALO route failed and OPENAI_API_KEY is not available for direct OpenAI fallback")
    cfg = _make_engine_config(HALO_DIRECT_OPENAI_MODEL)
    log_event("halo", "Starting HALO run via direct OpenAI", model=HALO_DIRECT_OPENAI_MODEL)
    return await run_engine_async([AgentMessage(role="user", content=halo_prompt)], cfg, HALO_TRACE_PATH, telemetry=False)

start = time.time()
try:
    log_event("halo", "Running HALO through Databricks OpenAI-compatible endpoint")
    halo_items = await _run_halo_with_databricks()
    halo_route = "databricks"
    log_event("halo", "Databricks HALO route completed", output_items=len(halo_items))
except Exception as databricks_exc:
    log_event("halo", "Databricks HALO route failed; trying direct OpenAI fallback", error_type=type(databricks_exc).__name__, error=str(databricks_exc)[:2000])
    print(type(databricks_exc).__name__, str(databricks_exc)[:2000])
    try:
        halo_items = await _run_halo_with_direct_openai()
        halo_route = "direct_openai"
        log_event("halo", "Direct OpenAI HALO route completed", output_items=len(halo_items))
    except Exception as openai_exc:
        halo_route = "failed"
        halo_items = []
        raise RuntimeError(
            "HALO failed through both Databricks and direct OpenAI. "
            f"Databricks error: {databricks_exc}\nDirect OpenAI error: {openai_exc}"
        ) from openai_exc

elapsed = time.time() - start
log_event("halo", "HALO run complete", route=halo_route, elapsed_seconds=round(elapsed, 1), output_items=len(halo_items))


In [ ]:
def _item_to_dict(item):
    try:
        return item.model_dump(mode="json")
    except Exception:
        return {"repr": repr(item)}


def _extract_halo_text(items: list[Any]) -> str:
    texts = []
    for item in items:
        data = _item_to_dict(item)
        obj = data.get("item", {}) if isinstance(data, dict) else {}
        if isinstance(obj, dict) and obj.get("role") == "assistant":
            content = obj.get("content")
            if isinstance(content, str):
                texts.append(content)
            elif isinstance(content, list):
                for part in content:
                    if isinstance(part, dict) and part.get("text"):
                        texts.append(part["text"])
    return "\n\n".join(texts).strip()


def clean_halo_handoff(report: str) -> str:
    normalized = re.sub(r"(?<!\n)(## Executive summary)", r"\n\n\1", report).strip()
    start = normalized.rfind("## Executive summary")
    if start == -1:
        raise ValueError("HALO output did not include the expected executive summary section.")
    handoff = normalized[start:].strip()
    required_headings = [
        "## Executive summary",
        "## Top 3 changes to implement first",
        "## Ranked recommendation table",
        "## Supporting diagnosis and evidence",
        "## Detailed recommendations",
        "## Insights by feedback source",
        "## Machine-readable summary",
    ]
    missing = [heading for heading in required_headings if heading not in handoff]
    if missing:
        raise ValueError(f"HALO handoff is missing required sections: {missing}")
    extra_top_level = [line for line in handoff.splitlines() if line.startswith("## ") and line not in required_headings]
    if extra_top_level:
        raise ValueError(f"HALO handoff included unexpected top-level sections: {extra_top_level}")
    return handoff


halo_payload = {
    "route": halo_route,
    "model": HALO_MODEL if halo_route == "databricks" else HALO_DIRECT_OPENAI_MODEL,
    "elapsed_seconds": elapsed,
    "trace_path": str(HALO_TRACE_PATH),
    "input_summary": _halo_input_summary_rows(),
    "items": [_item_to_dict(item) for item in halo_items],
}
HALO_OUTPUT_JSON.write_text(json.dumps(halo_payload, indent=2, default=str), encoding="utf-8")
log_event("artifacts", "Wrote HALO JSON payload locally", path=str(HALO_OUTPUT_JSON), item_count=len(halo_items))

raw_handoff = _extract_halo_text(halo_items)
if not raw_handoff:
    raw_handoff = json.dumps(halo_payload, indent=2, default=str)[:100000]
clean_handoff = clean_halo_handoff(raw_handoff)

HALO_HANDOFF_MD.write_text(clean_handoff.rstrip() + "\n", encoding="utf-8")
log_event("artifacts", "Wrote validated Codex handoff locally", path=str(HALO_HANDOFF_MD), handoff_chars=len(clean_handoff))
display(Markdown(clean_handoff))


In [ ]:
# Persist HALO artifacts to the same UC volume family used by skills.
halo_volume_dir = HALO_VOLUME_DIR
try:
    log_event("artifacts", "Persisting final HALO artifacts to UC volume", volume_dir=halo_volume_dir)
    dbutils.fs.mkdirs(halo_volume_dir)
    dbutils.fs.put(f"{halo_volume_dir}/halo_traces.jsonl", HALO_TRACE_PATH.read_text(encoding="utf-8"), True)
    dbutils.fs.put(f"{halo_volume_dir}/halo_output.json", HALO_OUTPUT_JSON.read_text(encoding="utf-8"), True)
    dbutils.fs.put(f"{halo_volume_dir}/codex_handoff.md", HALO_HANDOFF_MD.read_text(encoding="utf-8"), True)
    log_event("artifacts", "Persisted final HALO artifacts to UC volume", volume_dir=halo_volume_dir)
except Exception as exc:
    log_event("artifacts", "Could not persist final HALO artifacts", error_type=type(exc).__name__, error=str(exc)[:1000])

codex_handoff_markdown = HALO_HANDOFF_MD.read_text(encoding="utf-8")
log_event("done", "HALO notebook completed; displaying full codex_handoff.md", handoff_chars=len(codex_handoff_markdown))
display(Markdown(codex_handoff_markdown))


## Executive summary
The harness already (a) centralizes a strong production prompt in an MLflow “optimized_prompt” contract, (b) defines multiple UC deterministic tools (matchups/tendencies/arsenal/roster) plus a “≤8 tool calls” constraint and data-limitation wording, and (c) runs evaluation with both human feedback and an LLM judge (37 traces; 8 error traces; evidence shows both high-quality and failing behaviors).

Highest-value remaining gaps, as indicated by the eval rationales:
1) **When data is missing or tools fail/time out, the harness does not reliably produce a baseball-competent fallback strategy** (instead it often stops at “no data available” or gives non-actionable summaries).  
2) **Tool evidence is sometimes not fully incorporated into the recommendation** (answers list pitches but don’t convert to an actionable hitter plan; or give partially correct/incorrect numeric summaries).  
3) **Observability/guardrails are insufficient to prevent reliability failures** (examples explicitly mention “tools were timing out” / “Genie query terrible” / incorrect % sums), implying the execution loop needs clearer fallbacks, retries, and validation.

Gate status: based on repeated “needs strategic advantage” and “tools erroring/timing out” rationales, the system appears to pass some eval cases, but **fails frequently enough on strategy/fallback correctness that the overall optimization gate is not consistently satisfied.**

## Top 3 changes to implement first
1) **Add an explicit “Data Missing / Tool Failure Fallback” behavior contract + enforcement.**  
   *Rationale:* Multiple traces end with “no data available” or lack strategic advice even when some tool output exists (or when tools timed out). A deterministic fallback policy should always produce a hitter plan using the best available evidence (arsenal/repertoire + any partial tendency), and explicitly bound what is unknown.

2) **Add a “Recommendation must be evidence-to-strategy” validator in the harness.**  
   *Rationale:* Several low-score cases show relevant evidence retrieved but the response “stops short” of actionable strategy, or it lists pitches without attack-zone/timing logic. Enforce mapping: (retrieved pitch mix / zones / count filters) → (approach rules for early/2-strikes/sequence themes).

3) **Harden tool execution reliability + numeric sanity checks + truncation handling.**  
   *Rationale:* Evidence cites timeouts and “Genie query terrible,” plus numeric inconsistencies (e.g., % not summing to 100, or “data incorrect”). Add harness-level checks: retry policy, partial-success handling, and “don’t fabricate percentages / don’t imply 100%” rules with automatic phrasing constraints.

## Ranked recommendation table

| Rank | Recommendation | Impact | Confidence | Implementation effort | Evidence | Validation |
|---:|---|---|---|---|---|---|
| 1 | Implement a **mandatory fallback strategy** when matchup history returns empty / tools error / Genie fails | High (directly addresses lowest-rated failures) | High | Medium | Traces with “no data available” and low ratings for lacking actionable strategy; human feedback mentions tool timing-out | A/B in notebooks/10-HALOHarnessOptimization.ipynb: ensure zero “no strategy” outputs on failure scenarios |
| 2 | Add **evidence→strategy enforcement**: response must include (Attack zones + early/two-strike plan + what to avoid) when any tendency/arsenal evidence exists | High | High | Medium | Judge/human repeatedly: “need strategic advantage,” “summary incomplete,” “listing pitches but no actionable rec” | Automatic rubric checks in notebooks/09-Evaluation.ipynb; target higher “baseball_analysis_base” and fewer “mostly acceptable” downgrades |
| 3 | Add **tool reliability + output sanity checks** (timeouts/retries, partial output handling, no fabricated percentages, truncation-safe summaries) | Medium-High | Medium | Medium | Evidence mentions “tools timing out,” “Genie query terrible,” and numeric inconsistencies | Measure error_trace_count reduction and improved judge “data correctness” consistency |

## Supporting diagnosis and evidence

### Recurring harness-level failure modes (classified)

1) **No actionable strategy when data is missing or tools fail**  
   - Classification: **already-present-but-not-reliably-followed** (prompt says “If requested dataset missing… explicitly say ‘no [data] available…’ and do NOT imply stronger evidence”), but execution often stops there without a fallback plan.  
   - Evidence: traces where the assistant says it lacks data for a specific scenario and receives low “baseball_analysis_base / strategic advantage” scores; also explicit “Tools were timing out…” and “request took forever.”

2) **Evidence retrieved but recommendation is incomplete / not evidence-grounded**  
   - Classification: **implementation/observability defect** (no enforcement that retrieved tool evidence is fully converted into the required response structure + plan).  
   - Evidence: judge rationale: “summary is incomplete,” “stops short of offering strategic recommendation,” “no tactical advantage beyond listing pitches,” “should provide strategic recommendation.”

3) **Numeric/statistical inconsistencies** (percentages, denominators, mismatched totals; occasional incorrect data)  
   - Classification: **implementation/observability defect** (lack of numeric sanity checks and “do not fabricate” gating; plus potential truncation/partial parsing issues).  
   - Evidence: “Great answer but %s not adding up to 100% is weird”; “data incorrect” / “Genie query terrible it did not look for missing information”; “frequency wording … not reflected in summary.”

4) **Tool execution reliability issues (timeouts / incomplete Genie queries)**  
   - Classification: **implementation/observability defect**  
   - Evidence: explicit “Tools were erroring out,” “took forever,” “Genie query was terrible,” and several traces show high tool_count with poor completion.

### Mapping against harness context (what is missing vs not reliably followed)

- **Missing requirement:** a *hard requirement* that even on tool failure/empty result, the harness still outputs a complete **Recommendation** section with evidence-bounded baseball tactics (using arsenal/available tendency if possible).
- **Already present but not reliably followed:** “do NOT imply stronger evidence than exists” and “explicitly say ‘no [data] available…’” — present in the optimized prompt, but the same traces still underperform on strategic completeness and sometimes wording correctness.
- **Implementation/observability defect:** there is no visible mechanism (from outcomes) that validates: (a) recommendation section exists, (b) contains attack zones/timing, (c) uses retrieved evidence comprehensively, (d) numeric statements are consistent with tool outputs.

## Detailed recommendations

### Behavior contract
#### Prompt
- Keep the existing prompt contract, but add **two explicit clauses**:
  1) **Failure fallback clause (mandatory):**  
     When any tool returns empty/timeout/error, the assistant must still produce the full markdown structure and a **50–75 word at-bat plan** using whatever evidence exists (e.g., arsenal repertoire + any partial tendencies), and must explicitly label unknowns.
  2) **Evidence-to-plan clause (mandatory):**  
     The Recommendation must reference at least **one** of: (a) count-specific tendency, (b) repertoire/pitch mix from arsenal, (c) observed matchup history metrics—whichever is available—and convert it into a hitter action (zones/timing/what to avoid). If none exist, provide a generic but still baseball-specific plan (e.g., “work counts/attack strike zone” with a clear “no data available” limitation).

#### Skills
- Update skills definitions (not just prompt) to reflect the fallback contract:
  - In each skill’s workflow, add a final step: **“If matchup_history empty or tool failure: run fallback evidence synthesis using arsenal/tendencies already retrieved; otherwise state ‘no data available’ but still output a generic evidence-bounded plan.”**
- Ensure skill selection does not “early exit” after empty rows.

### Runtime implementation
#### Tools
- Add harness-level tool orchestration policies (in the Databricks harness):
  - **Retry/timeouts:** for deterministic UC functions, implement bounded retries on transient errors; if timeouts occur, proceed with partial evidence rather than returning a “no data” response.
  - **Partial-success handling:** if one tool succeeds and others fail, the response must still use the successful tool outputs to populate Data collected + Recommendation.
- Add a “tool output integrity” normalization step:
  - For tendency outputs, validate expected fields exist before using them.
  - For any computed frequencies, enforce phrasing constraints (“frequency_pct is within this filtered set” vs “% of all pitches”) per the prompt rule.

#### Control flow
- Enforce a deterministic response assembly pipeline:
  1) Collect evidence objects from each successful tool call.
  2) Decide which evidence categories are available: {matchup_history, count_tendency, arsenal/pitch_mix, roster/context, genie fallback}.
  3) Render markdown sections using only available categories.
  4) If matchup_history empty or tool errors: run **fallback synthesis** (must still fill Pitcher Approach + Recommendation).
- Add an “early stop prevention” guard:
  - Do not allow the model to conclude after stating missing data; always require Recommendation generation step.

#### Routing
- Prefer fewer tool calls when possible (≤8 already), but introduce routing rules:
  - If count-specific tendency tool fails, fall back to repertoire/arsenal + any available general tendencies.
  - If handedness lookup fails, do not guess handedness; route to name-based lookup first (as per skills).

### Output contract
#### Response schema
- Enforce strict compliance with the prompt’s required markdown structure:
  - `# At-Bat Assistant Assessment`
  - `## Data collected` (≤50 words, no raw row dumps)
  - `## Pitcher Approach` (≤200 words total; must include count subheadings if count data exists)
  - `## Recommendation` (50–75 words; must be actionable and evidence-tied)
- Add a validation check: if any evidence exists, Recommendation must include at least one zone cue phrase and one sequence/timing cue phrase; otherwise it fails the harness gate.

### Observability and evals
#### Observability
- Add explicit trace logging fields (as span attributes) for:
  - which evidence categories were successfully retrieved,
  - tool failure reasons (timeout/error type),
  - counts/filters used (season, count, base state, handedness),
  - whether fallback mode was activated.
- Capture “evidence coverage score” (boolean per category) so you can correlate failure modes with judge scores.

#### Evals
- Extend evaluation suite (notably notebook 09 and harness optimization notebook 10):
  - **Fallback correctness tests:** simulate empty tool outputs and tool errors; assert response still has an actionable Recommendation.
  - **Evidence-to-plan rubric:** check presence of (attack zones + timing/avoidances) when evidence exists.
  - **Numeric sanity tests:** flag likely fabricated percentages (e.g., “35%” when tool output omitted percent fields; or “sums to 100%” claims).

## Insights by feedback source
- **runtime_trace (MLflow execution traces):**  
  Shows both strong outputs (e.g., Skubal-style detailed repertoire with compliant language) and repeated failures where tools returned nothing/errored and the assistant produced non-actionable “no data” responses or incomplete summaries (e.g., Fried tendencies listing without strategic advantage).
- **human_feedback:**  
  Calls out tool reliability issues (“Tools were timing out…”, “Genie query terrible”) and content quality gaps (“need strategic advantage,” “there’s enough here to provide some feedback,” “I want to see what they define as breaking balls”).
- **llm_judge_feedback:**  
  Provides the clearest patterns: failures for **lack of tactical recommendation**, non-compliance with baseball-language guideline, and relevance/analysis gaps even when evidence is available (“stops short,” “no strategic augmentation,” “completely unacceptable”).
- **deterministic_eval_feedback:**  
  Zero examples (explicitly stated in your input summary).
- **mlflow_assessment_feedback:**  
  Zero examples (explicitly stated in your input summary).
- **optimized_prompt:**  
  Contains strong rules for evidence grounding, data limitation wording, tool usage, and mandatory response format—yet execution outcomes suggest these rules aren’t enforced as hard gates.
- **uc_function_signatures:**  
  Present and detailed; indicates deterministic tools exist (matchup/tendency/arsenal/roster) and therefore the fallback can be built from known tool families rather than “no data” alone.
- **aligned_judge_memory:**  
  Zero examples (explicitly stated).
- **generated_skills:**  
  Skills exist for matchup-game-plan, count-and-handedness-tendencies, runner-state-tendencies, arsenal-and-pitch-mix, team-lineup-matchups, genie-fallback-analysis, and data-limitation-handling—however workflows likely need stronger “fallback still outputs full strategy” enforcement.
- **harness_config:**  
  Present in the input summary but no concrete settings were enumerated in the prompt you provided; still, errors/timeouts in traces imply harness orchestration needs improvement.

## Machine-readable summary
```json
{
  "top_priorities": [
    "Add mandatory fallback strategy when matchup history is empty or tools error/time out; still produce full Recommendation (50-75 words) evidence-bounded to available tools.",
    "Implement evidence→strategy enforcement in the harness: if any evidence category exists, Recommendation must include attack-zone + timing/avoidance cues and explicitly tie to retrieved evidence.",
    "Harden tool execution reliability (retry/partial-success routing) and add numeric/output integrity sanity checks to prevent incorrect or fabricated frequency statements."
  ],
  "evidence_sources": [
    "runtime_trace (tool failures/empty results correlate with low strategic scores)",
    "human_feedback (timeouts/Genie failures; strategic-advantage missing)",
    "llm_judge_feedback (repeated rationales about lack of actionable recommendation and incomplete summaries)",
    "optimized_prompt (good contract exists but not enforced reliably)",
    "generated_skills (present but workflows need stronger fallback + evidence coverage steps)"
  ],
  "validation_plan": [
    "Introduce eval cases that force empty/timeout tool outputs; assert Recommendation is still present and actionable (rubric-based).",
    "Run evidence-to-plan rubric checks: when count/arsenal/matchup evidence exists, response must include zone cues and a sequence/timing plan.",
    "Add numeric sanity validators: flag percentage claims not backed by tool fields and detect inconsistent denominators or impossible sums."
  ],
  "manual_review_required": [
    "Review wording changes to ensure fallback strategies remain consistent with baseball interpretation requirements (no mph fabrication; no implied percentages; correct phrasing of frequency_pct semantics)."
  ]
}
```
